# Tutorial 7a — Two-model coupling I: what a coupling *is*

Estimated time: 20-25 minutes. No optional backend required.

> **Module 7 comes in three parts.** Each answers one question and can be done in one
> sitting.
>
> | | Question | Needs |
> |---|---|---|
> | **7a (you are here)** | What does it *mean* to couple two models, and what does the default sampler do with it? | nothing extra |
> | **7b** | How is the coupled distribution actually sampled, and can I trust the sampler? | PyMC, for the later steps |
> | **7c** | What questions can I ask it — including inferring one model's parameters from another model's measurement? | PyMC |

## What you need installed

| Need | Why | Install |
|---|---|---|
| `bayesian-metamodeling` + `[tutorials]` | The framework, plus numpy/matplotlib for the plots | `pip install -e ".[tutorials]"` |
| **Nothing else** | Coupling itself needs neither PyMC nor SBI — the "Why no PyMC" section explains why, and the reason is worth knowing | — |

## What you'll be able to do afterwards

- **Say what a coupling *is*, in probability** — write down the density you sampled from,
  not just the JSON you typed.
- **Read a coupled scatter plot** and explain its shape from the spec's `sigma`.
- **Recognise the failure that looks like success**: a "posterior" that is really a prior
  with a transform applied to it.

## How you'll know you got it

You can predict, before running anything, whether the coupling's *source* variable will come
out narrower than its prior — and say what the answer tells you about which method ran.


## Where this sits in the series

Everything so far produced **one model's** view of the world. This is where two of them
start talking.

| Tutorial | What it gave you | What module 7 does with it |
|---|---|---|
| T1 | ran a sweep: inputs in, outputs out, one row per design point | the raw material a surrogate is fitted to |
| T3, T4 | the spec contract, and where the design points go | the same spec discipline, now for a *metamodel* |
| T5, T6 | a **surrogate**: a probabilistic stand-in for one simulator | the things being coupled |
| **7a, 7b, 7c** | — | **a statement that two models' variables refer to the same quantity, machinery to sample the consequences, and the questions you can then ask** |
| T8 | — | three models in a chain: does the story survive? |
| T9 | — | your own pipeline, end to end |

The gap module 7 fills is specific. A surrogate tells you what *one* model believes. Nothing
so far lets one model's belief constrain another's. That is what a coupling is for, and it is
the reason this framework exists rather than just fitting models one at a time.


## Vocabulary, briefly

You met all of these in T1-T6. Here they are in one place, in the sense module 7 uses them,
so nothing below depends on remembering a definition from three tutorials ago.

| Term | In one sentence | Where it came from |
|---|---|---|
| **Partial model** | One team's model of one piece of the system — a simulator, an ODE, a fit. | T1, T2 |
| **Sweep** | Running a model at many input combinations and recording the outputs. | T1, T4 |
| **Surrogate** | A fast probabilistic stand-in for a model, fitted to its sweep. Given inputs it returns a *distribution* over outputs, not a single number. | T5, T6 |
| **Prior** | What you believed about a quantity before looking at this data — usually a plausible range. | T5 |
| **Posterior** | What you believe *after* folding in the evidence. Narrower than the prior when the evidence was informative. | T5 |
| **Coupling** | Your assertion that two models' variables refer to the same physical quantity, to within a tolerance σ. | **7a, here** |
| **Joint distribution** | One probability distribution over *all* the variables at once, rather than one per model. | **7a, here** |
| **Propagation vs inference** | Two different things `meta sample` can do. Telling them apart is the point of 7a. | **7a, here** |


## Why this tutorial matters

Two models were built independently. One says what `y` should be given `x`; the other
says what `z` should be given `y`. Left alone, the metamodel knows of no relationship
between their variables at all.

A **coupling** is the sentence you add to make them talk: *whatever these two models are,
this pair of quantities is the same thing, up to a disagreement of width σ.* It is a
modelling choice you make and defend — the framework will not discover it for you, and it
will not check it. Two cells from here you will see exactly what that sentence does to the
probability; first, why the shape of the answer depends on which method you ask for.

What the framework does *with* the assertion depends on which of two methods you ask for,
and **that difference is the main lesson of this notebook**:

- **Step 1 runs the default, `--method propagate`.** It draws every variable from its
  prior and then overwrites the coupling's target: `C := y + Normal(0, σ)`. It never
  evaluates a surrogate. The scatter you are about to see is *forward propagation of a
  coupling assumption* — the answer to "if I believe this coupling, what does it imply
  downstream?" Calling it a posterior would be a misnomer, and Step 2 prints the
  framework's own record saying so.
- **Step 3 shows `--method joint`**, which samples the actual joint density — priors,
  couplings *and* surrogate likelihoods together. There both ends of a coupling move and
  the surrogates are conditioned on. That is inference.

Both are useful. Confusing them is the mistake this notebook exists to prevent.

T8 next chains three couplings and asks whether the story generalises past two. The
full-scale version — four fitted surrogates, real couplings, `--method joint` — is
`projects/tcr_signaling/notebooks/03_metamodel_inference.ipynb`, which T9 points you to
at the end.

## Why no PyMC, no SBI — and what that tells you

Tutorial 5 needed PyMC. Tutorial 6 needed SBI and PyTorch. This one needs neither, even
though it is the most *probabilistic* notebook so far. Worth a minute, because the answer
is not "we simplified it for the tutorial".

**The libraries were for fitting, not for coupling.** PyMC and SBI were doing one job in
T5/T6: taking a table of simulator runs and turning it into a probabilistic model. That is
the expensive, specialised part, and it is finished by the time you get here — the result
is a saved artifact.

**Coupling happens above that layer.** A coupling is an assertion relating variables, and
the pieces needed to sample its consequences are ordinary arithmetic:

- draw from a normal distribution — `numpy` has that;
- apply a transform like `C = y` or `z = 1.0·C + 0.0` — arithmetic;
- evaluate a log-density and accept or reject a proposed step — arithmetic.

So the metamodel layer imports numpy and nothing heavier. You can check that claim rather
than trust it: nothing under `src/bayesian_metamodeling/meta/` imports `pymc`, `torch` or
`sbi`.

**Then what is `"ppl_backend": "pymc"` in the spec?** Nearly nothing, today. It is a
declaration of which probabilistic-programming library the graph is *intended* to be
compiled for, if that ever gets built. Right now `compile_metamodel` accepts the string,
checks it is one of two allowed values, and stores it. Its only effect on any number you
will see is that choosing `"numpyro"` multiplies the coupling noise by 1.05 in the
propagate path — a deliberate nudge so the two labels do not produce byte-identical output.

*(Jargon, once, so the term is not a wall: a **PPL** — probabilistic programming language —
is a library like PyMC or Stan where you declare random variables and it works out how to
sample them. This framework does not use one at the metamodel layer.)*

**The lesson to take, which outlives this notebook:** a field in a config file that looks
like a dependency may be a statement of intent. Read what the code does with a setting
before you conclude anything from its name. You will meet this again — `pymc_gp`, the
backend you used in T5, is not a Gaussian process either.

The flip side is the real constraint, and it arrives in Step 4: `--method joint` conditions
on the surrogates, so it needs *fitted* ones. The libraries are not needed to couple; they
are needed to have something worth coupling.


## Coupling, written as probability

Everything below follows from one equation, built in three small steps.

**Step A — two models that have never met.** Model A gives you a belief about `y`; some
other model gives you a belief about `C`. Knowing nothing that relates them, the joint
belief is the product:

$$p(y, C) \;=\; p(y)\,p(C)$$

Multiplying is exactly what "independent" means. Draw from it and you get a round blob:
learning `y` tells you nothing about `C`.

**Step B — the assertion, written as a score.** You claim these two quantities are *the
same thing*, measured by two different models, which may disagree a little. Write the claim
as a function that scores how well any pair `(y, C)` satisfies it — high when they agree,
falling away as they drift apart:

$$\phi(y, C) \;=\; \mathrm{Normal}\!\left(C;\; y,\; \sigma\right)$$

σ is the whole content of the claim: *how far apart may these two models be before I would
call it a contradiction?* Small σ is a strong claim; large σ is a weak one. This is your
scientific judgement, and the framework will neither check it nor discover it for you.

> **Why a score and not a conditional distribution.** It is tempting to read that formula
> as `p(C | y)` and conclude that `C` is now *generated* from `y`. That is not what a
> coupling is here. It is an extra **factor** — a soft constraint multiplied into what you
> already believed — so `C` keeps its own prior and the constraint pulls on both variables.
> This is exactly the difference the next cell measures. Reading the coupling as "C is
> produced from y" is, in effect, choosing `--method propagate` without realising you chose
> anything.

**Step C — multiply it in.** The coupled joint is the product of what you believed before
and what you have now asserted:

$$p(y, C) \;\propto\; \underbrace{p(y)\,p(C)}_{\text{what each model believed alone}} \;\times\; \underbrace{\phi(y, C)}_{\text{your coupling}}$$

Both priors survive. That is the point of writing the coupling as a factor rather than as
`p(C \mid y)`: nothing is replaced, something is added.

That third factor is large where `C ≈ y` and tiny elsewhere, so it *reshapes* the round
blob into a ridge along the diagonal. **That reshaping is the entire mechanism.** A
coupling adds a factor to a product; sampling then respects it.

Two consequences fall straight out of the algebra, and the cell below draws both:

1. **It constrains both variables.** The factor mentions `y` and `C` symmetrically, so
   conditioning on it should narrow `y` as well — not just place `C` next to `y`.
2. **A cheaper shortcut exists**, and it is what `--method propagate` does: draw `y` from
   its prior, then *set* `C := y + Normal(0, σ)`. That reproduces the ridge, but it quietly
   drops `p(C)` from the product and leaves `y` at exactly its prior width. Fast, useful,
   and **not the same distribution**.

Hold that distinction — the next cell makes it visible, and it is the main lesson here.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# The spec's numbers: y ~ Normal(0,1); C has no prior of its own, so it picks up the
# default Normal(0,1); the coupling is gaussian_link(y -> C) with identity transform.
SIGMA = 0.15
grid = np.linspace(-3.2, 3.2, 320)
Y, C = np.meshgrid(grid, grid, indexing="ij")

def lognorm(v, mu, sd):
    return -0.5 * ((v - mu) / sd) ** 2 - np.log(sd)

prior_log = lognorm(Y, 0, 1) + lognorm(C, 0, 1)          # Step A: independent
coupled_log = prior_log + lognorm(C, Y, SIGMA)            # Step C: x the coupling factor
prior_d = np.exp(prior_log - prior_log.max())
coupled_d = np.exp(coupled_log - coupled_log.max())

# What --method propagate actually generates: y from its prior, then C overwritten.
_rng = np.random.default_rng(0)
y_prop = _rng.normal(0.0, 1.0, 4000)
C_prop = y_prop + _rng.normal(0.0, SIGMA, 4000)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)
for ax, field, title in (
    (axes[0], prior_d, "A. Two models, no coupling\n$p(y)\\,p(C)$"),
    (axes[1], coupled_d, f"C. Coupled joint (the truth)\n$\\times\\,N(C; y, {SIGMA})$"),
):
    ax.contourf(Y, C, field, levels=14, cmap="viridis")
    ax.set_title(title); ax.set_xlabel("y"); ax.set_ylabel("C"); ax.set_aspect("equal")
axes[2].scatter(y_prop, C_prop, s=3, alpha=0.25, color="tab:orange")
axes[2].set_title("What --method propagate draws\n(y from prior, C := y + noise)")
axes[2].set_xlabel("y"); axes[2].set_ylabel("C")
axes[2].set_xlim(-3.2, 3.2); axes[2].set_ylim(-3.2, 3.2); axes[2].set_aspect("equal")
plt.show()

# The joint in the middle panel is Gaussian, so its exact width is available in closed
# form -- no sampling needed. Precision matrix of the product, then invert it.
prec = np.array([[1 + 1 / SIGMA**2, -1 / SIGMA**2],
                 [-1 / SIGMA**2, 1 + 1 / SIGMA**2]])
cov = np.linalg.inv(prec)
sd_joint = np.sqrt(np.diag(cov))

print(f"sigma = {SIGMA}\n")
print(f"{'':<26}{'sd(y)':>9}{'sd(C)':>9}{'corr':>9}")
print(f"{'prior, uncoupled':<26}{1.0:>9.3f}{1.0:>9.3f}{0.0:>9.3f}")
print(f"{'coupled joint (exact)':<26}{sd_joint[0]:>9.3f}{sd_joint[1]:>9.3f}"
      f"{cov[0,1] / (sd_joint[0]*sd_joint[1]):>9.3f}")
print(f"{'propagate (sampled)':<26}{y_prop.std():>9.3f}{C_prop.std():>9.3f}"
      f"{np.corrcoef(y_prop, C_prop)[0,1]:>9.3f}")
print(f"\nsd(y): the coupling narrows it to {sd_joint[0]:.3f} in the true joint,")
print(f"       but propagate leaves it at {y_prop.std():.3f} - its prior width, untouched.")


**Read the third column first: the correlation is nearly the same in both.** Both the true
joint and the propagate shortcut produce a tight diagonal ridge. If you only ever looked at
a scatter plot, you would not be able to tell them apart — which is exactly why this
mistake survives in real projects.

**Now read `sd(y)`.** In the true coupled joint it is about 0.71 — the exact value, since
that joint is Gaussian and the cell inverts its precision matrix rather than sampling it.
Under propagate it comes back at essentially 1.0, its prior width, differing only by
sampling noise, because propagate never touched it.

That single number is the whole difference:

- **The true joint learns about `y`.** The assertion "`C` is `y` up to 0.15" is information
  about `y` too, and `p(C)` — which prefers `C` near 0 — pushes back on `y` through the
  coupling. Both ends move.
- **Propagation does not.** It generates `y`, then *decorates* it with `C`. Information
  flows one way, downstream only, and `p(C)` is silently discarded.

So: propagate answers **"if I take this coupling as given, what does it imply downstream?"**
The joint answers **"what should I believe about *both* quantities, given the coupling and
everything else I assumed?"** The first is forward uncertainty propagation. Only the second
is inference.

Both are legitimate and the default is the first. Reporting the first as a posterior is the
error this notebook exists to prevent — and the rest of the notebook is you catching the
framework in the act, with its own stored artifacts as evidence.


## What are we actually coupling? (read this before Step 1)

This tutorial uses two **pre-built example surrogate artifacts** that ship with the repo.
Open them — they are four lines each:

- `examples/coupled/artifacts/surrogate_A.artifact.json` — declares `inputs: ["x"]`,
  `outputs: ["y"]`. Model A maps `x → y`.
- `examples/coupled/artifacts/surrogate_B.artifact.json` — declares `inputs: ["y"]`,
  `outputs: ["z"]`. Model B maps `y → z`.

Two things about them matter, and both are easy to skim past:

1. **They are signatures, not fits.** Each declares which variables go in and which come
   out, and carries no `backend_payload` — no trained model whatsoever. That is enough for
   `meta build` to assemble the graph, and enough for `--method propagate`, which never
   evaluates a surrogate. It is *not* enough for `--method joint`, which refuses out loud
   — you will trigger that refusal on purpose in Step 3.
2. **The coupled variable `C` belongs to neither model.** The spec
   `tutorials/specs/metamodel.two_model.pymc.json` declares four variables (`x, y, z, C`)
   but gives priors only to `x` and `y`. `C` appears in no surrogate and in no prior: it
   is a bare name that exists only as the target of the `gaussian_link`, and the sampler
   silently gives it a default `Normal(0, 1)` rather than complaining.

So be precise about what this spec demonstrates: **the coupling primitive**, with `C`
standing in for "some other model's quantity" — not a handshake between two fitted
models. The next cells print the built graph so you can see that rather than take it on
faith.

A coupling target with no prior and no model behind it is a warning sign worth learning to
recognise in your own specs. Nothing warns you, the sampler is happy, and the samples look perfectly
healthy. The only defence is reading the compiled graph, which is exactly what Step 1
does.

You will edit this spec's `sigma` field in the checkpoint below.

## Step 1: Build the graph, then propagate the coupling through it

In [ ]:
# Cross-platform setup (Windows / macOS / Linux) — no shell, no PYTHONPATH prefix.
# Find the repo root so `src/` is importable, then load the shared tutorial helpers.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "src" / "bayesian_metamodeling").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from bayesian_metamodeling.tutorial import bootstrap, run_mm_cli, run_tool

root = bootstrap()  # chdir to repo root + ensure src/ on sys.path (idempotent)
ROOT = root
print("Repo root:", root)

# Steps 1-3 need no optional backend at all. Steps 4-5 fit a real surrogate, which does.
try:
    import pymc  # noqa: F401
    PYMC_AVAILABLE = True
except ImportError:
    PYMC_AVAILABLE = False
    print("\nNote: PyMC is absent, so Steps 4-5 (fitting a real surrogate) will skip.")
    print("Steps 1-3 — the coupling itself — run without it. See the prerequisites table.")


In [ ]:
run_mm_cli('meta', 'build', 'tutorials/specs/metamodel.two_model.pymc.json')
run_mm_cli('meta', 'sample', 'tutorials/specs/metamodel.two_model.pymc.json', '--draws', '200', '--tune', '100', '--chains', '2', '--seed', '123')

In [ ]:
# `meta build` wrote the assembled model to disk as plain JSON. Open it - this IS the
# model: a flat list of factors, with no library-specific machinery in it.
import json
from pathlib import Path

IR_NAME = "tutorial_two_model_coupling"
_ir_files = sorted((root / "tmp/metamodel_ir").glob("*/ir.json"), key=lambda p: p.stat().st_mtime)
ir_payload = ir_path = None
for _p in reversed(_ir_files):
    _cand = json.loads(_p.read_text())
    if _cand.get("name") == IR_NAME:
        ir_payload, ir_path = _cand, _p
        break
assert ir_payload is not None, f"No IR named {IR_NAME!r} — run the `meta build` cell above."

print("IR file:", ir_path.relative_to(root))
print("variables:", [v["name"] for v in ir_payload["variables"]])
print(f"\n{len(ir_payload['factors'])} factors:")
for f in ir_payload["factors"]:
    if f["kind"] == "prior":
        print(f"  prior                 {f['variable']} ~ {f['distribution']}")
    elif f["kind"] == "coupling":
        print(
            f"  coupling              {f['coupling_type']:<24}"
            f" {f['source']} -> {f['target']}  transform={f['transform']} sigma={f.get('sigma')}"
        )
    else:
        print(f"  surrogate_likelihood  {f['surrogate_ref']}: {f['inputs']} -> {f['outputs']}")

_priored = {f["variable"] for f in ir_payload["factors"] if f["kind"] == "prior"}
_all_vars = {v["name"] for v in ir_payload["variables"]}
print("\nvariables with an explicit prior :", sorted(_priored))
print("variables without one (default N(0,1)):", sorted(_all_vars - _priored))

**What you just printed is the model.** `meta build` does one job: read the spec and write
out a flat list of **factors** — the individual pieces that get multiplied together to form
the joint density from the section above. `meta sample` then reads that list and draws from
it. Nothing in the list is specific to any library, which is why it is plain JSON you can
read.

(The codebase calls this list the **IR**, for *intermediate representation* — "intermediate"
because it sits between the spec you wrote and the sampler that consumes it. You will meet
the term in paths like `tmp/metamodel_ir/`.)

Six factors here, in three groups:

- **2 priors** — `x ~ N(0,1)` and `y ~ N(0,1)`. `z` and `C` have none and fall back to a
  default `N(0,1)`. For `z` that never shows (a deterministic coupling pins it); for `C`
  it is the prior that gets drawn and then immediately overwritten.
- **2 couplings** — the `gaussian_link` that shapes the cloud you are about to see, and
  the `deterministic_transform` that locks `z` to `C`.
- **2 surrogate likelihoods** — `surrogate_A: [x] → [y]` and `surrogate_B: [y] → [z]`.
  They are in the graph. Under `--method propagate` they contribute **exactly zero**:
  `CompiledMetaModel.evaluate_log_prob` in `meta/compiler.py` `continue`s past any
  likelihood factor whose surrogate is not loaded, and the propagate path calls it with
  `surrogates={}`.

That is the distinction worth carrying with you: **the factor list says what the model
*is*; the method says which factors actually get *evaluated*.** Two runs over the very same
model can therefore mean different things — which is why every stored result records the
method that produced it.

Watch `x` in particular. It has a prior, it is surrogate A's input, and under propagate it
is a pure prior draw that influences nothing at all — the only thing that could have tied
it to `y` is surrogate A's likelihood, which is switched off. It will still appear in the
samples file with a full column of perfectly good numbers. *"A variable was sampled" is not
"a variable was informed."*

**Coupling kinds in this spec.** Look at the `couplings` array in
`tutorials/specs/metamodel.two_model.pymc.json` — there are two, and they behave in
genuinely different ways:

- `kind: "gaussian_link"` between `y` and `C`, with `sigma: 0.15`. **Probabilistic.**
  Reads as: "`y` and `C` should agree, with Gaussian noise of width σ = 0.15 around the
  transform." Larger σ = looser coupling = wider cloud. This is what *shapes* the scatter.
- `kind: "deterministic"` between `C` and `z`, with affine transform `α=1, β=0`. **Hard
  constraint.** Reads as `z = α·C + β`, exactly, no noise.

Watch which variable behaves which way: `C` is shaped by `y` through a probabilistic link
(so `C` ends up a noisy copy of `y`), and `z` is then locked to `C` (so `z` is *identical*
to `C` in the samples).

**How "deterministic" is actually enforced — it is not a penalty.** Neither sampler ever
*proposes* `z` and then checks it. `propagate` computes `z = 1.0·C + 0.0` directly after
drawing `C` (`meta/sampling.py`); `joint` treats `z` as a **derived** quantity recomputed
from its source at every step (`derived_variables` in `meta/joint_sampling.py`, whose
docstring notes that proposing it freely "would be rejected essentially always"). The
`_DETERMINISTIC_PENALTY = -1e6` in `compiler.py` exists so that an off-surface point
evaluates as effectively impossible if you ask for its log-density directly — but no
sampling path ever visits one. The observable consequence: the next cell prints
`max|z − C|`, and it is exactly `0`, not `1e-9`.

(Implementation note: under the hood, *every* non-`deterministic` coupling kind compiles to
the same Gaussian-noise primitive — `equality_soft` and `gaussian_link` are the same code
path in `compiler.py`. T8 uses `gaussian_link` too. We picked one term for both notebooks.)

## Step 2: Visualize the propagated samples (graphic)

**Predict before you look at the scatter.** The spec gives `y ~ Normal(0,1)`, `C` an
implicit `Normal(0,1)`, and a `gaussian_link` with σ = 0.15 from `y` to `C`. Under
`--method propagate`, `C` is drawn from its prior and then *overwritten* with
`y + Normal(0, 0.15)`.

Commit to an answer for each of these before you run the cell:

1. Without the coupling the cloud is round. With σ = 0.15, how tight is the diagonal — and
   what number do you expect `std(y − C)` to print?
2. If you changed σ to 1.0, would the diagonal **vanish** or just **widen**?
3. The one most people get wrong: does the coupling make `std(y)` **smaller** (surely `y`
   is now constrained by `C`?), unchanged, or larger? And what about `std(C)`?

Hold those predictions, then run.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# --- pick the samples this notebook just produced ---------------------------------
# The registry holds samples from every metamodel ever sampled in this repo, so sort by
# `created_at` (NOT by UUID-hex order, which is essentially random) and take the most
# recent run whose `inference_data.json` names THIS metamodel. Matching on the IR name
# rather than on "has y and C" keeps T8's three-model samples out of the picture.
IR_NAME = "tutorial_two_model_coupling"
reg = json.loads((root / "tmp/metamodel_samples_registry.json").read_text())


def _abs(p: str) -> Path:
    q = Path(p)
    return q if q.is_absolute() else root / q


chosen = None
for _created, _sid, _meta in sorted(
    ((m.get("created_at", ""), s, m) for s, m in reg.items()), reverse=True
):
    _ds_raw, _inf_raw = _meta.get("samples_dataset_path"), _meta.get("inference_data_path")
    if not _ds_raw or not _inf_raw:
        continue
    _ds_path, _inf_path = _abs(_ds_raw), _abs(_inf_raw)
    if not _ds_path.is_file() or not _inf_path.is_file():
        continue
    _info = json.loads(_inf_path.read_text())
    if _info.get("name") == IR_NAME:
        chosen = (_ds_path, _info)
        break
if chosen is None:
    raise RuntimeError(f"No metamodel samples for {IR_NAME!r} — run Step 1 above first.")
dataset_path, info = chosen

vars_ = json.loads(dataset_path.read_text())["variables"]
y = np.asarray(vars_["y"], dtype=float).reshape(-1)
c = np.asarray(vars_["C"], dtype=float).reshape(-1)
z = np.asarray(vars_["z"], dtype=float).reshape(-1)

# Read sigma from the spec rather than hardcoding it, so this cell keeps telling the
# truth after you edit the spec in the checkpoint below.
spec = json.loads((root / "tutorials/specs/metamodel.two_model.pymc.json").read_text())
sigma = float(next(cp for cp in spec["couplings"] if cp["kind"] == "gaussian_link")["sigma"])

# The framework's own record of what it did. Read this BEFORE believing any plot.
print(f"{dataset_path.parent.name}/inference_data.json says:")
print(f"    method               = {info['method']!r}")
print(f"    surrogates_evaluated = {info['surrogates_evaluated']}")
print(f"    (spec's gaussian_link sigma = {sigma})")

# A "no coupling" baseline: independent draws from the priors p(y) = p(C) = Normal(0,1).
rng = np.random.default_rng(42)
y_prior = rng.normal(0, 1, size=len(y))
c_prior = rng.normal(0, 1, size=len(y))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)

axes[0].scatter(y_prior, c_prior, alpha=0.25, s=18, color="tab:gray")
axes[0].set_title("WITHOUT coupling: independent priors  (p(y)·p(C))")
axes[1].scatter(y, c, alpha=0.25, s=18, color="tab:blue")
axes[1].set_title(f"WITH gaussian_link σ={sigma}: propagated (NOT a posterior)")
for ax in axes:
    ax.set_xlabel("y")
    ax.set_ylabel("C")
    ax.grid(True, alpha=0.3)
    ax.axline((0, 0), slope=1, color="k", linestyle=":", alpha=0.5, label="y = C")
    ax.legend(loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nWITHOUT coupling: corr(y, C) = {np.corrcoef(y_prior, c_prior)[0, 1]:+.3f}   (≈ 0)")
print(
    f"WITH coupling   : corr(y, C) = {np.corrcoef(y, c)[0, 1]:+.3f}   "
    f"(closed form 1/sqrt(1+σ²) = {1 / np.sqrt(1 + sigma**2):.3f})"
)
print(
    f"WITH coupling   : std(y - C) = {np.std(y - c, ddof=1):.3f}   "
    f"(the spec's σ = {sigma})"
)
print("\nWhere the width went:")
print(
    f"    std(y) = {y.std(ddof=1):.3f}  — the coupling's SOURCE, still at its prior scale 1.0"
)
print(
    f"    std(C) = {c.std(ddof=1):.3f}  — the TARGET; std(C)/std(y) = "
    f"{c.std(ddof=1) / y.std(ddof=1):.3f} vs sqrt(1+σ²) = {np.sqrt(1 + sigma**2):.3f}"
)
print(
    f"    max|z - C| = {np.max(np.abs(z - c)):.1e}  — the deterministic coupling is exact, "
    "not approximate"
)

**Read those last three numbers again — they are the whole point of Step 2.**

`std(y)` did not move. The coupling's **source** is exactly as wide as its prior said it
was; nothing about "`y` and `C` agree" was allowed to inform `y`. And `std(C)` came out
*wider* than its own `Normal(0,1)` prior, by a factor of `sqrt(1 + σ²)` — because propagate
constructs the target as `y + noise`, which **adds** a variance rather than removing one.

That is the opposite of what conditioning on agreement should do. Real inference on this
constraint shrinks **both** variables: agreement is evidence, and evidence narrows. Here
one variable is untouched and the other got wider, so what you are looking at is not
evidence being used — it is an assumption being pushed downstream, in one direction.

Keep this asymmetry as your field test:

> **If a coupling's source still has exactly its prior width, no inference happened.**

Step 3 runs the same test on a sampler that passes it. T8 turns it into a full uncertainty
budget across a chain of three couplings.

The deterministic coupling shows the other extreme: `max|z − C|` is exactly zero — no
tolerance, no noise. `z` was never sampled at all; it was computed.

## Scientific checkpoint (active learning)

The single best way to internalise coupling is to vary `sigma` and watch what changes.

1. Open `tutorials/specs/metamodel.two_model.pymc.json`, find the `gaussian_link` between
   `y` and `C`, note its `sigma: 0.15`.
2. Change σ to **0.5**, save, and re-run Step 1's two cells and the scatter cell.
3. Repeat for σ = **1.0**, then σ = **3.0**, and (optional) σ = **0.05** — almost as tight
   as deterministic.

**Answer key.** Under `propagate` these relationships are exact, not empirical. `C = y + ε`
with `ε ~ N(0, σ)` independent of `y ~ N(0,1)`, so

- `std(y − C) = σ` exactly, and
- `corr(y, C) = 1 / sqrt(1 + σ²)` exactly.

Run the next cell for the table, then grade your printouts against it.

- *At what σ does the coupling stop having a visible effect?* Later than you expect. At
  σ = 1.0 the correlation is still around 0.7 and the diagonal is obvious. You need σ ≈ 3 —
  a coupling three times wider than the prior it is constraining — before the cloud looks
  round. The diagonal degrades smoothly; it never truly vanishes.
- *What does `corr(y, C)` tell you about the strength of agreement?* Exactly what σ tells
  you, re-expressed — it is a monotone function of σ alone. Under propagate, that
  correlation is a property of the **spec**, not of the data and not of the surrogates. Sit
  with that: a high correlation here is not evidence that two models agree, it is a
  restatement of what you asserted.
- *Compare σ to the printed `std(y − C)`.* They match to sampling error at every σ. That is
  a config number doing exactly what it promises, and it is how a field in a JSON file
  becomes something physical.

**Reset σ to 0.15** when you are done, so later tutorials see the spec they expect. (The
self-check reads σ from the spec, so it will not lie to you if you forget — but the number
it prints will tell on you.)

In [ ]:
# The answer key, computed rather than asserted. Both columns are exact consequences of
# `C = y + Normal(0, sigma)` with y ~ Normal(0, 1) — no sampling involved.
import numpy as np

print(f"{'sigma':>7}   {'corr(y,C) = 1/sqrt(1+s^2)':>25}   {'std(y-C) = sigma':>16}")
for s in (0.05, 0.15, 0.5, 1.0, 3.0):
    print(f"{s:7.2f}   {1 / np.sqrt(1 + s**2):25.3f}   {s:16.2f}")
print("\nAt sigma = 3 the coupling is 3x wider than the prior it constrains, and the")
print(
    f"correlation it induces is only {1 / np.sqrt(1 + 3.0**2):.2f} — that is where the cloud"
    " starts to look round."
)

## Recap: what 7a established

- **A coupling is an assertion you make**, not a fact the framework discovers.
  `gaussian_link(y → C, σ)` says these two quantities are the same up to noise of width σ.
  σ is the disagreement you are willing to tolerate, and you can measure it in the output as
  `std(y − C)`.
- **A coupling is a factor, not a generative step.** It multiplies into a product of priors
  and reshapes it. Written as `p(C | y)` instead, it would only ever push information
  downstream — which, as it happens, is exactly what the default method does.
- **`deterministic` is not "σ = 0 with a penalty".** Its target is never sampled; it is
  computed from its source, every step. `max|z − C|` is exactly 0.
- **`--method propagate` is not inference.** It draws priors and rewrites the coupling's
  target. The source keeps its prior width, the target comes out *wider*, and no surrogate is
  evaluated. The artifact admits it: `method: "prior_propagation"`,
  `surrogates_evaluated: false`.
- **The factor list is the model; the method is what gets evaluated.** Two runs over the same
  model can mean different things, which is why every stored result records how it was made.

**Next — 7b** takes the same spec and samples the coupled joint properly: what changes when
both ends of a coupling are allowed to move, how to tell whether the sampler actually
explored anything, and why a surrogate fitted to noiseless data quietly breaks the whole
thing.


## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `No metamodel samples for 'tutorial_two_model_coupling'` | Step 1 did not run in this kernel | Re-run the bootstrap cell and Step 1. `run_mm_cli` always executes from the repo root, so the spec's relative paths resolve. |
| `std(y − C)` is far from the σ you just set | You edited the spec but did not re-run `meta build` + `meta sample` | Re-run Step 1's cells after every spec edit — the sampler reads the file, not your intentions. |
| `corr(y, C)` lower than you expected | σ is larger than you think | Under propagate the correlation is `1/sqrt(1+σ²)` and nothing else. Check the spec, not the surrogates. |
| Sampling finished instantly and you expected slow | `propagate` runs no MCMC at all | Correct behaviour. Only `--method joint` runs a chain, and only it evaluates surrogates. |
| `x` looks like noise and correlates with nothing | It is surrogate A's input, and surrogate A is switched off under propagate | Also correct. Nothing connects `x` to `y` except a likelihood that is never evaluated. |

**The failure worth internalising:** a metamodel that samples cleanly and returns the prior
is *not* an error — no exception, no warning, plausible output. Under `--method propagate`
it is the **guaranteed** behaviour for every variable that is not a coupling target, which
is exactly why comparing sampled width against prior width has to become automatic. It is
the only thing that catches this.

## Final check: the coupling was applied, and the artifact admits it was propagation

Reads σ out of `tutorials/specs/metamodel.two_model.pymc.json` and the latest samples for
`tutorial_two_model_coupling`, then asserts four things. Each is a claim that could actually
fail on a notebook that ran but taught nothing:

1. `std(y − C) ≈ σ` — the spec's σ *is* the observed disagreement width.
2. `corr(y, C) ≈ 1/sqrt(1+σ²)` — the exact propagate relation, at whatever σ the spec
   currently holds, with a tolerance scaled to the sample size.
3. `max|z − C| == 0` — the deterministic coupling is exact, not approximate.
4. `std(y) ≈ 1.0` — the coupling's **source** is untouched. This is the fingerprint of
   propagation, and it is the assertion that would fail if someone quietly made `propagate`
   start doing inference.

Plus: the stored artifact records `method: "prior_propagation"` and
`surrogates_evaluated: false`.


In [ ]:
# Self-check: the propagate path applied the coupling exactly as the spec says, and the
# stored artifact admits that no surrogate was evaluated.
import json as _json
from pathlib import Path as _P

import numpy as _np

_spec = _json.loads((root / "tutorials/specs/metamodel.two_model.pymc.json").read_text())
_sigma = float(next(_c for _c in _spec["couplings"] if _c["kind"] == "gaussian_link")["sigma"])
_prior_y = float(
    next(_p for _p in _spec["priors"] if _p["variable"] == "y")["distribution"]["scale"]
)

_reg = _json.loads((root / "tmp/metamodel_samples_registry.json").read_text())


def _abs2(p: str) -> _P:
    q = _P(p)
    return q if q.is_absolute() else root / q


_chosen = None
for _created, _sid, _meta in sorted(
    ((m.get("created_at", ""), s, m) for s, m in _reg.items()), reverse=True
):
    _dsr, _infr = _meta.get("samples_dataset_path"), _meta.get("inference_data_path")
    if not _dsr or not _infr:
        continue
    _dsp, _infp = _abs2(_dsr), _abs2(_infr)
    if not _dsp.is_file() or not _infp.is_file():
        continue
    _info = _json.loads(_infp.read_text())
    if _info.get("name") == "tutorial_two_model_coupling":
        _chosen = (_json.loads(_dsp.read_text()), _info)
        break
assert _chosen is not None, (
    "No samples for 'tutorial_two_model_coupling' — Step 1's `meta sample` didn't run."
)
_ds, _info = _chosen
_y = _np.asarray(_ds["variables"]["y"], dtype=float).reshape(-1)
_c = _np.asarray(_ds["variables"]["C"], dtype=float).reshape(-1)
_z = _np.asarray(_ds["variables"]["z"], dtype=float).reshape(-1)
_n = int(_y.size)

# (1) the spec's sigma IS the disagreement width.
_disagree = float(_np.std(_y - _c, ddof=1))
assert abs(_disagree - _sigma) < 0.2 * _sigma, (
    f"std(y - C) = {_disagree:.4f} but the spec says sigma = {_sigma}. Either the "
    "gaussian_link did not apply, or these samples predate your spec edit — re-run "
    "`meta build` + `meta sample`."
)

# (2) the exact propagate relation, at whatever sigma the spec currently holds.
_corr = float(_np.corrcoef(_y, _c)[0, 1])
_corr_exact = float(1.0 / _np.sqrt(1.0 + _sigma**2))
# Tolerance = 4 standard errors of a Pearson r at this n, floored at 0.03. Scaling with n
# and rho keeps the check honest at sigma = 0.05 and at sigma = 3.0 alike.
_corr_tol = max(0.03, 4.0 * (1.0 - _corr_exact**2) / _np.sqrt(max(_n - 1, 1)))
assert abs(_corr - _corr_exact) < _corr_tol, (
    f"corr(y, C) = {_corr:.4f}, but propagate implies 1/sqrt(1+sigma^2) = "
    f"{_corr_exact:.4f} +- {_corr_tol:.4f} at sigma = {_sigma}."
)

# (3) deterministic means exact, not close.
_zmax = float(_np.max(_np.abs(_z - _c)))
assert _zmax == 0.0, (
    f"max|z - C| = {_zmax:.3e} — a deterministic coupling must be exact. If this is a "
    "small non-zero number, the target is being sampled instead of derived."
)

# (4) the SOURCE never moved: the fingerprint of propagation.
_sd_y = float(_y.std(ddof=1))
assert abs(_sd_y - _prior_y) < 0.15 * _prior_y, (
    f"std(y) = {_sd_y:.4f} vs its prior scale {_prior_y}. Under --method propagate the "
    "coupling's source must keep its prior width; if it shrank, this dataset came from "
    "joint sampling, not propagation."
)

# (5) the artifact says what it is.
assert _info["method"] == "prior_propagation", (
    f"inference_data.json records method = {_info['method']!r}, expected 'prior_propagation'."
)
assert _info["surrogates_evaluated"] is False, (
    "inference_data.json records surrogates_evaluated = True, which --method propagate "
    "can never do."
)

print(
    f"\n[T7a self-check OK] sigma={_sigma}: std(y-C)={_disagree:.3f}; "
    f"corr(y,C)={_corr:.3f} vs exact {_corr_exact:.3f} (tol {_corr_tol:.3f}); "
    f"max|z-C|={_zmax:.1e}; std(y)={_sd_y:.3f} vs prior {_prior_y} (source untouched); "
    f"method={_info['method']!r}, surrogates_evaluated={_info['surrogates_evaluated']}."
)